# ASMSA: Prepare and check input files

**Next steps**
- [tune.ipynb](tune.ipynb): Perform initial hyperparameter tuning for this molecule
- [train.ipynb](train.ipynb): Use results of previous tuning in more thorough training
- [md.ipynb](md.ipynb): Use a trained model in MD simulation with Gromacs

In [ ]:
# Go to the working directory, created in 01-setup.ipynb, and read in things defined there

%cd ~/trpcage
exec(open('inputs.py').read())

In [ ]:
#avoid TF to consume GPU memory
import tensorflow as tf

tf.config.set_visible_devices([], 'GPU')
tf.config.list_logical_devices()

import torch

In [ ]:
import mdtraj as md
import numpy as np
import urllib.request
import asmsa
import os
import shutil
import glob

import re
import tensorflow as tf
import gromacs as gmx
import gromacs.fileformats as gf
import matplotlib.pyplot as plt
import numpy as np

from collections import defaultdict

In [ ]:
from asmsa.helpers import extract_dihedral_indices_local, extract_atom_indices

## Prepare input files

#### Generate important files

   * **Density of additional internal coordinates**: how many randomly sampled distances from all atom-to-atom one atom should appear in average
   * **Inputs file generation**: file to be used in the next notebooks
   * **pdb2gmx**: convert a PDB file into a GROMACS topology and generate a coordinate file, topology file and index file

In [ ]:
gmx.pdb2gmx(f=conf, ignh=True,p=topol,n=index,o=gro,water='tip3p',ff='amber99sb-ildn')

### Sanity checks:
 * **Check**: if everything in your trajectory is fine, before proceeding

In [ ]:
# Load the trajectory, it should report expected numbers of frames and atoms/residua

tr = md.load(traj,top=conf)
idx=tr[0].top.select("name CA")

# for trivial cases like Ala-Ala, where superposing on CAs fails
#idx=tr[0].top.select("element != H")

tr.superpose(tr[0],atom_indices=idx)

In [ ]:
# Visual check, all frames should look "reasonable"

# Because of different conventions of numbering atoms in proteins,
# PDB file {conf} and the trajectory {traj} can become inconsistent, and this would appear here 
# as rather weird shapes of the molecule

import nglview as nv

v = nv.show_mdtraj(tr)
v.clear()
v.add_representation("licorice")
v.add_representation("cartoon")
v

## Datasets preprocessing

### Shuffle configurations in trajectory

In [ ]:
# shuffle the trajectory so the configurations are dispersed across all datasets
np.random.shuffle(tr.xyz)

### Select proportions to divide the trajectory

In [ ]:
# - set proportions for train, validation and test datasets
# - proportions must be equal to 1 when added together
train = .7
validation = .15
test = .15

assert train + validation + test == .9999999999999999 or 1

tr_i = len(tr) * train
X_train = tr.slice(slice(0,int(tr_i)))

va_i = len(tr) * validation
X_validate = tr.slice(slice(int(tr_i),int(tr_i)+int(va_i)))

te_i = len(tr) * test
X_test = tr.slice(slice(int(tr_i)+int(va_i),len(tr)))

X_train.xyz.shape, X_validate.xyz.shape, X_test.xyz.shape

### Divide the trajectory


In [ ]:
X_train.save_xtc('train.xtc')
X_validate.save_xtc('validate.xtc')
X_test.save_xtc('test.xtc')

#### Recovery

In [ ]:
# eventual recovery

X_train = md.load_xtc('train.xtc',conf)
X_validate = md.load_xtc('validate.xtc',conf)
X_test = md.load_xtc('test.xtc',conf)


### Compute RMSD between
   * **train x validation** trajectory and filter similar structures in train trajectory
   * **train x test** trajectory and filter similar structures in train trajectory
   * **test x validation** trajectory and filter similar structures in test trajectory

In [ ]:
# get RMSD from train trajectory compared to validation trajectory
gmx.select(s=conf,on='backbone.ndx',select='Backbone')

In [ ]:
gmx.rms(s=conf,f='train.xtc',f2='validate.xtc',n='backbone.ndx',m='trainxval_rmsd.xpm')

In [ ]:
# load the RMDS matrix
txv = gf.XPM('trainxval_rmsd.xpm')
txv.array.shape

In [ ]:
# minima per row -- for each configuration in train, how far is the nearest one from validation
txv_min = np.min(txv.array,axis=1)
txv_min.shape

In [ ]:
plt.hist(txv_min,bins=50)
plt.show()

In [ ]:
# drop similar structures (to validation trajectory) in train trajectory to avoid dataset being biased
txv_difference = 0.05

train_tr = X_train[np.argwhere(txv_min > txv_difference).flat]
train_tr.xyz.shape

In [ ]:
train_tr.save_xtc('tmp_train.xtc')
gmx.rms(s=conf,f='tmp_train.xtc',f2='test.xtc',n='backbone.ndx',m='trainxtest_rmsd.xpm')

In [ ]:
txt = gf.XPM('trainxtest_rmsd.xpm')
txt.array.shape

In [ ]:
txt_min = np.min(txt.array,axis=1)
txt_min.shape

In [ ]:
plt.hist(txt_min,bins=50)
plt.show()

In [ ]:
# ... one more time with test trajectory & test x validation...
txt_difference = 0.05

x_train = train_tr[np.argwhere(txt_min > txt_difference).flat]
x_train.save_xtc('x_train.xtc')

In [ ]:
# test x validation
gmx.rms(f='test.xtc',f2='validate.xtc',s=conf,n='backbone.ndx',m='testxvalidate_rmsd.xpm')

In [ ]:
txv = gf.XPM('testxvalidate_rmsd.xpm')
txv.array.shape

In [ ]:
txv_min = np.min(txv.array,axis=1)
txv_min.shape

In [ ]:
plt.hist(txv_min,bins=50)
plt.show()

In [ ]:
# ... one more time with test trajectory & test x validation...
txv_difference = 0.05

x_test = X_test[np.argwhere(txv_min > txv_difference).flat]
x_test.save_xtc('x_test.xtc')

In [ ]:
'''
# skip thorough RMS
! ln train.xtc x_train.xtc
! ln test.xtc x_test.xtc
'''

In [ ]:
%mkdir -p 00.computations
!mv x_train.xtc x_test.xtc validate.xtc 00.computations

#### Recovery

In [ ]:
# recovery

x_train = md.load('./00.computations/x_train.xtc', top=conf)
x_test = md.load('./00.computations/x_test.xtc', top=conf)
x_val = md.load('./00.computations/validate.xtc', top=conf)

x_all = md.load(traj, top=conf)

#### Save computetd geometries

In [ ]:
# get shapes of filtered trajectories that are to be used as datasets

trajs = [x_train, x_val, x_test, x_all]
x_train.xyz.shape, x_val.xyz.shape, x_test.xyz.shape, x_all.xyz.shape

In [ ]:
# reshuffle the geometries to get frame last so that we can use vectorized calculations later on
geoms = [ np.moveaxis(t.xyz,0,-1) for t in trajs]
print ([ g.shape for g in geoms ])

In [ ]:
# save geometries

tf.data.Dataset.from_tensor_slices(geoms[0]).save('01.datasets/geoms/train')
tf.data.Dataset.from_tensor_slices(geoms[1]).save('01.datasets/geoms/validate')
tf.data.Dataset.from_tensor_slices(geoms[2]).save('01.datasets/geoms/test')
tf.data.Dataset.from_tensor_slices(geoms[3]).save('01.datasets/geoms/X_all')

### Internal coordinates computation

Exercise the ASMSA library on your input. Just check everything appears to work.

There are multiple options that can be combined:
* use traditional internal coordinates (bond distances, angles, and dihedrals) or not
* include additional distances between atoms that may not be bound to express protein folding state more directly
   * dense (all-to-all) atom distances, feasible for very small peptides only
   * sparse atom distances (only some pairs are chosen)
   
* We save the computed internal coordinates for training, and a feature extraction model here, therefore everything in the other notebooks should work too.


#### Compute atom indexes : 
* Backbone
* ON (bacbone Oxigens, Nitrogens)
* Polar atoms
* Alpha Carbons
* Alpha-Beta Carbons

In [ ]:
exctract = extract_atom_indices(conf)

backbone = exctract["backbone"]
nC = exctract["nC"]
alpha = exctract["alpha"]
alphabeta = exctract["alphabeta"]

print(f'Backbone({len(backbone)}): {backbone}')
print(f'nC({len(nC)}): {nC}')
print(f'Alpha C ({len(alpha)}): {alpha}')
print(f'Alpha and Beta ({len(alphabeta)}): {alphabeta}')

#### Extract:
* Bonds
* Angles
* Dihedrals
* Dihedrlas: phi-psi only

In [ ]:
bonds = np.array([[backbone[i], backbone[i+1]] for i in range(len(backbone) - 1)])
angles = np.array([[backbone[i], backbone[i+1], backbone[i+2]] for i in range(len(backbone) - 2)])
dih = np.array([backbone[i:i+4] for i in range(len(backbone) - 3)])

In [ ]:
dih_all = extract_dihedral_indices_local(conf)
print(len(dih_all))

#### Compute distances:
* 1) **Choose** the atom selection (e.g atoms = ON): the atoms will be considered for the random pick with the density selection.
  2) **If** "sparse" choose the density.
  3) **If** "dense" all the atoms will be considered.
  

In [ ]:
sparse_dists = asmsa.NBDistancesSparse(geoms[0].shape[0], density=nb_density)
sparse_dists_customs = asmsa.NBDistancesSparse(geoms[0].shape[0], density=5,  atoms = nC)
dense_dists = asmsa.NBDistancesDense(geoms[0].shape[0])
dense_dists_customs = asmsa.NBDistancesDense(geoms[0].shape[0], atoms=nC)

#### Build:

In [ ]:
#mol_small=asmsa.Molecule(pdb=conf,top=topol,fms=[dense_dists])

#mol = asmsa.Molecule(pdb=conf,top=topol,ndx=index,fms=[sparse_dists])

#To custom tou mol use either `top` or `bonds/angles/diheds` can be specified, not both

mol_customs =asmsa.Molecule(pdb=conf,n_atoms=geoms[0].shape[0],
                           diheds=dih_all,
                           fms=[dense_dists_customs]) 

In [ ]:
mol_model = mol_customs.get_model()

example_input = torch.randn((*geoms[0].shape[:2],1))

In [ ]:
print(f'Sparse dists = {len(sparse_dists.bonds)}')
print(f'Sparse dists customs = {len(sparse_dists_customs.bonds)}')
print(f'Dense dists = {len(dense_dists.bonds)}')
print(f'Dense dists customs = {len(dense_dists_customs.bonds)}')

In [ ]:
#mol_customs.describe_features() #No if you have defined bonds/angles/dihe

#### Save the features (molecule) model

In [ ]:
mol_model = mol_customs.get_model()

example_input = torch.randn((*geoms[0].shape[:2],1))
traced_script_module = torch.jit.trace(mol_model, example_input)

traced_script_module.save('features.pt')

#### Compute the interanal coordinates now

In [ ]:
intcoords = [ mol_customs.intcoord(g).T for g in geoms]
print(
    [ g.shape for g in geoms ],
    [ i.shape for i in intcoords ]
)

In [ ]:
[train,validate,test, X_all] = intcoords

In [ ]:
# validate the saved model -- should yield nearly 0.

test_from_model = mol_model(torch.from_numpy(geoms[2])).numpy()
np.max(test - test_from_model.T)

In [ ]:
test_from_model.shape

In [ ]:
# normalize training set
train_mean = np.mean(train,axis=0)
train -= train_mean
train_scale = np.std(train,axis=0)
train /= train_scale

In [ ]:
# normalize test and validation sets
test -= train_mean
test /= train_scale
validate -= train_mean
validate /= train_scale
X_all -= train_mean
X_all /= train_scale

#### Save internal coordinates as datasets which can be loaded in **train.ipynb** and **tune.ipynb** notebooks 

In [ ]:
# save for usage in tune/train/test phase

tf.data.Dataset.from_tensor_slices(train).save('01.datasets/intcoords/train')
tf.data.Dataset.from_tensor_slices(validate).save('01.datasets/intcoords/validate')
tf.data.Dataset.from_tensor_slices(test).save('01.datasets/intcoords/test')
tf.data.Dataset.from_tensor_slices(X_all).save('01.datasets/intcoords/X_all')

np.savetxt('01.datasets/intcoords/mean.txt',train_mean)
np.savetxt('01.datasets/intcoords/scale.txt',train_scale)

# Density of the conformational space

- Sample the training trajectory randomly
- For each point in the trajectory:
  - calculate RMSD to all points in the sample
  - pick some number $n$ of nearest ones
  - calculate the _density_ at this point as $$ d = \sum_{i=1}^n e^{-d_i} / n $$  i.e. the nearer the sample points are, the higher the density
 
Altogether, $d$ roughly corresponds to the probability that the molecule during simulation ends up in this area of the conformational space.

In [ ]:
sample_size = 5000
x_train = md.load('x_train.xtc', top=conf)
tr_sample = x_train[np.random.choice(len(x_train),sample_size,False)]
tr_sample.save('sample.xtc')

In [ ]:
gmx.rms(f='x_train.xtc',f2='sample.xtc',s=conf,n='backbone.ndx',m='sample_rmsd.xpm')

In [ ]:
rms = gf.XPM('sample_rmsd.xpm')

#### Visual check to verify the sample size is representative
- typically, not many distances should be less than 0.1 nm and more than 1 nm 
(the latter depends on the molecule, can be more for e.g. big disordered proteins)
- the histogram should be semi-smooth

In [ ]:
plt.hist(rms.array.flatten(),bins=50)
plt.show()

In [ ]:
k_nearest = 200
rms_sort = np.sort(rms.array.astype(np.float32))
erms = np.exp(-rms_sort[:,:k_nearest])
dens = (np.sum(erms,axis=1)-1.) / (erms.shape[1] - 1)

#### Histogram of densities
- quite high number of points should fall above 0.8, those are low energy basins
- the interval [0.5, 1.0] should be reasonably covered
- on the contrary, too many points below 0.4 would indicate either insufficient sampling above or too sparse trajectory

In [ ]:
plt.hist(dens,bins=20)
plt.show()

In [ ]:
len(dens),len(x_train)

In [ ]:
np.savetxt('datasets/train_density.txt',dens)